# Notebook 7: Quantization & KV Cache

Why do RLMs exist? Because **long context is expensive**. This notebook explains:
- Why attention is O(n^2) and what that means for long documents
- What the KV cache is and why it eats VRAM
- How quantization (FP16 → INT8 → INT4) shrinks models
- Why RLMs sidestep the long-context problem entirely

## The Core Problem

A 100,000-token document requires the model to compute attention between every pair of tokens. That's 10 billion operations — and it gets worse as context grows.

## Attention is O(n^2)

In a transformer, every token attends to every other token. If your context has `n` tokens:
- Compute cost: O(n^2) — quadratic growth
- Memory for attention: O(n^2)
- This means doubling context length → 4x the compute

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Show how compute grows with context length
context_lengths = np.array([1000, 2000, 4000, 8000, 16000, 32000, 64000, 128000])
compute = context_lengths ** 2  # O(n^2)
memory = context_lengths  # KV cache grows linearly

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Compute cost
axes[0].plot(context_lengths / 1000, compute / 1e9, 'r-o', linewidth=2)
axes[0].set_xlabel('Context Length (thousands of tokens)')
axes[0].set_ylabel('Relative Compute (billions of ops)')
axes[0].set_title('Attention Compute Cost: O(n^2)')
axes[0].fill_between(context_lengths / 1000, compute / 1e9, alpha=0.2, color='red')
axes[0].annotate('4K context', xy=(4, 16/1e3), fontsize=9)
axes[0].annotate('128K context\n(1024x more compute!)', xy=(100, 128**2/1e3), fontsize=9, ha='right')

# KV cache memory
kv_bytes = context_lengths * 2 * 32 * 128 * 2  # 2 (K,V) * 32 layers * 128 dim * 2 bytes
axes[1].plot(context_lengths / 1000, kv_bytes / 1e9, 'b-o', linewidth=2)
axes[1].set_xlabel('Context Length (thousands of tokens)')
axes[1].set_ylabel('KV Cache Size (GB)')
axes[1].set_title('KV Cache Memory: O(n)')
axes[1].fill_between(context_lengths / 1000, kv_bytes / 1e9, alpha=0.2, color='blue')

plt.tight_layout()
plt.show()

print("Key insight: A 128K context uses 1,024x more compute than a 4K context!")
print("This is why RLMs are valuable — they keep each call's context small.")

## What is the KV Cache?

During text generation, the model stores **key-value pairs** for every token it has seen. This is the KV cache.

- **Purpose:** Avoid recomputing attention for previous tokens
- **Cost:** Grows linearly with context length
- **Impact:** Uses significant VRAM — often more than the model weights themselves

**Formula:** `KV Cache Size ≈ 2 × layers × hidden_dim × context_length × bytes_per_param`

For a 7B parameter model with 32 layers, 4096 hidden dim, at FP16:
- 4K context: ~0.5 GB
- 32K context: ~4 GB
- 128K context: ~16 GB (more than the model itself!)

## What is Quantization?

Quantization reduces the **precision** of model weights to shrink the model:

| Precision | Bits per Parameter | 7B Model Size | Quality |
|-----------|-------------------|---------------|---------|
| FP32 | 32 bits | ~28 GB | Full |
| FP16 | 16 bits | ~14 GB | Nearly identical |
| INT8 | 8 bits | ~7 GB | Slight quality loss |
| INT4 | 4 bits | ~3.5 GB | Noticeable for complex tasks |

**Our setup:** Qwen3-1.7B with AWQ (4-bit) uses ~1.5 GB VRAM, fitting easily on your 8GB GPU.

## Quantization Formats

Three common formats:

| Format | Full Name | Best For | How It Works |
|--------|-----------|----------|-------------|
| **GPTQ** | GPT-Quantization | GPU inference | Post-training quantization using calibration data |
| **AWQ** | Activation-Aware Weights | GPU inference | Protects important weights based on activation patterns |
| **GGUF** | GPT-Generated Unified Format | CPU/GPU mixed | Flexible format used by llama.cpp, works without GPU |

**We use AWQ** because it's fast on GPU and preserves quality well for small models.

## How RLMs Sidestep the Problem

Instead of processing 100K tokens in one expensive pass, RLMs break it into many small, cheap calls:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def vanilla_cost(n_tokens, max_context=4096):
    """Cost of stuffing everything into one prompt."""
    if n_tokens > max_context:
        return float('inf')  # Can't fit!
    return n_tokens ** 2  # Attention cost

def rlm_cost(n_tokens, chunk_size=2000, overhead=1.5):
    """Cost of RLM recursive processing."""
    n_chunks = max(1, n_tokens // chunk_size)
    # Each chunk costs chunk_size^2, plus overhead for coordination
    return n_chunks * (chunk_size ** 2) * overhead

context_lengths = np.array([1000, 2000, 4000, 8000, 16000, 32000, 64000])

vanilla_costs = [vanilla_cost(n) for n in context_lengths]
rlm_costs = [rlm_cost(n) for n in context_lengths]

fig, ax = plt.subplots(figsize=(10, 6))

# Cap vanilla at reasonable display value
vanilla_display = [min(v, 5e9) for v in vanilla_costs]
ax.plot(context_lengths / 1000, vanilla_display, 'r-o', label='Vanilla (stuff in prompt)', linewidth=2)
ax.plot(context_lengths / 1000, rlm_costs, 'g-o', label='RLM (recursive chunks)', linewidth=2)

# Mark where vanilla exceeds context window
ax.axvline(x=4, color='red', linestyle='--', alpha=0.5, label='Typical context window limit (4K)')
ax.annotate('Vanilla FAILS here\n(exceeds context window)', 
            xy=(4.5, 3e9), fontsize=10, color='red')

ax.set_xlabel('Document Length (thousands of tokens)')
ax.set_ylabel('Relative Compute Cost')
ax.set_title('Vanilla vs RLM: Compute Cost by Document Length')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print("Key insight: RLMs scale linearly with document length.")
print("Vanilla LLMs scale quadratically AND hit a hard context window limit.")

In [ ]:
def vram_calculator(model_params_B, quant_bits, context_length, n_layers, hidden_dim):
    """Estimate VRAM usage for a model + KV cache."""
    # Model weights
    bytes_per_param = quant_bits / 8
    model_vram_gb = model_params_B * 1e9 * bytes_per_param / 1e9
    
    # KV cache (2 for K and V, FP16)
    kv_vram_gb = 2 * n_layers * hidden_dim * context_length * 2 / 1e9
    
    # Activation memory (rough estimate)
    activation_gb = 0.5  # ~500MB overhead
    
    total = model_vram_gb + kv_vram_gb + activation_gb
    
    print(f"Model: {model_params_B}B params @ {quant_bits}-bit = {model_vram_gb:.1f} GB")
    print(f"KV Cache: {context_length:,} tokens = {kv_vram_gb:.2f} GB")
    print(f"Activations: ~{activation_gb} GB")
    print(f"Total: {total:.1f} GB")
    print(f"{'Fits on 8GB GPU!' if total < 8 else 'TOO LARGE for 8GB GPU'}")
    return total

print("=== Qwen3-1.7B AWQ (our setup) ===")
vram_calculator(1.7, 4, 4096, 24, 2048)

print("\n=== Qwen3-8B FP16 (full precision) ===")
vram_calculator(8, 16, 4096, 32, 4096)

print("\n=== Llama 70B INT4 ===")
vram_calculator(70, 4, 4096, 80, 8192)

## Key Takeaways

1. **Attention is O(n^2)** — doubling context length quadruples compute cost
2. **KV cache** stores key-value pairs for every token — grows linearly, eats VRAM
3. **Quantization** (FP16->INT8->INT4) shrinks models by 2-8x with modest quality loss
4. **AWQ** is our choice — fast, quality-preserving, GPU-optimized
5. **RLMs sidestep the problem** — many small calls instead of one huge call
6. **The tradeoff:** RLMs trade latency (many calls) for capability (unbounded context)

## What's Next?

Notebook 8 explores **function calling and tool use** — how `llm_query()` connects to the broader pattern of LLMs using external tools.